In [1]:
# gcs_test.py
import german_compound_splitter.comp_split as comp_split
import os

# --- НАСТРОЙКИ ---
# Путь к вашему словарю
DICTIONARY_PATH = "U:/voothi/20241223170748-token-extraction/20250826000433-test/german.dic"

# Слова, которые мы хотим протестировать
WORDS_TO_TEST = [
    "Ausbildungserfahrung",
    "DSL-Dienste",
    "Informationstechnik",
    "Arbeitspapiere" # Добавил для демонстрации make_singular
]

# --- СКРИПТ ---

def run_test():
    """Запускает изолированный тест GCS."""
    if not os.path.exists(DICTIONARY_PATH):
        print(f"ОШИБКА: Файл словаря не найден по пути: {DICTIONARY_PATH}")
        return

    print("Загрузка словаря GCS...")
    ahocs = comp_split.read_dictionary_from_file(DICTIONARY_PATH)
    print("Словарь загружен.\n")

    for word in WORDS_TO_TEST:
        print(f"--- Анализ слова: '{word}' ---")

        # --- Тест 1: Режим по умолчанию (как в вашем коде для существительных) ---
        # only_nouns=True, make_singular=True
        # Это "безопасный" режим, ищет только существительные и приводит их к единственному числу.
        dissection1 = comp_split.dissect(word, ahocs, make_singular=True, only_nouns=True)
        result1 = comp_split.merge_fractions(dissection1)
        print(f"  Режим 1 (По умолчанию, only_nouns=True, make_singular=True):")
        print(f"    -> Результат: {result1}")

        # --- Тест 2: Режим --gcs-only-nouns-false ---
        # only_nouns=False, make_singular=True
        # Это "агрессивный" режим, ищет любые части речи (предлоги, наречия и т.д.).
        dissection2 = comp_split.dissect(word, ahocs, make_singular=True, only_nouns=False)
        result2 = comp_split.merge_fractions(dissection2)
        print(f"  Режим 2 (--gcs-only-nouns-false):")
        print(f"    -> Результат: {result2}")

        # --- Тест 3: Режим --gcs-combine-noun-modes ---
        # Эмуляция объединения результатов из Теста 1 и Теста 2.
        combined_result = sorted(list(set(result1) | set(result2)))
        print(f"  Режим 3 (Эмуляция --gcs-combine-noun-modes):")
        print(f"    -> Результат: {combined_result}")

        # --- Тест 4: Демонстрация make_singular=False ---
        # Сравним, что будет, если не приводить к единственному числу.
        dissection4 = comp_split.dissect(word, ahocs, make_singular=False, only_nouns=True)
        result4 = comp_split.merge_fractions(dissection4)
        print(f"  Режим 4 (only_nouns=True, make_singular=False):")
        print(f"    -> Результат: {result4}")
        
        print("-" * (len(word) + 20))


if __name__ == "__main__":
    run_test()

Загрузка словаря GCS...
Loading data file - U:/voothi/20241223170748-token-extraction/20250826000433-test/german.dic
Словарь загружен.

--- Анализ слова: 'Ausbildungserfahrung' ---
Dissect compound:  Ausbildungserfahrung
  Режим 1 (По умолчанию, only_nouns=True, make_singular=True):
    -> Результат: ['Ausbildung', 'Erfahrung']
Dissect compound:  Ausbildungserfahrung
  Режим 2 (--gcs-only-nouns-false):
    -> Результат: ['ausBildung', 's', 'Erfahrung']
  Режим 3 (Эмуляция --gcs-combine-noun-modes):
    -> Результат: ['Ausbildung', 'Erfahrung', 'ausBildung', 's']
Dissect compound:  Ausbildungserfahrung
  Режим 4 (only_nouns=True, make_singular=False):
    -> Результат: ['Ausbildungs', 'Erfahrung']
----------------------------------------
--- Анализ слова: 'DSL-Dienste' ---
Dissect compound:  DSL-Dienste


IndexError: list index out of range

In [5]:
# gcs_test.py (версия с патчем от бага и новым словом)
import german_compound_splitter.comp_split as comp_split
import os

# --- НАСТРОЙКИ ---
# Путь к вашему словарю
DICTIONARY_PATH = "U:/voothi/20241223170748-token-extraction/20250826000433-test/german.dic"

# Слова, которые мы хотим протестировать
WORDS_TO_TEST = [
    "Ausbildungserfahrung",
    "Informationstechnik",
    "Unternehmenssoftware", # <--- ДОБАВЛЕНО НОВОЕ СЛОВО
    "Arbeitspapiere",
    "DSL-Dienste" # Это слово вызывает баг в GCS
]

# --- СКРИПТ ---

def run_test():
    """Запускает изолированный тест GCS с патчем от бага."""
    if not os.path.exists(DICTIONARY_PATH):
        print(f"ОШИБКА: Файл словаря не найден по пути: {DICTIONARY_PATH}")
        return

    print("Загрузка словаря GCS...")
    ahocs = comp_split.read_dictionary_from_file(DICTIONARY_PATH)
    print("Словарь загружен.\n")

    for word in WORDS_TO_TEST:
        print(f"--- Анализ слова: '{word}' ---")

        # --- Тест 1: Режим по умолчанию (как в вашем коде для существительных) ---
        result1 = []
        try:
            # ПАТЧ: Оборачиваем вызов в try/except, чтобы поймать IndexError
            dissection1 = comp_split.dissect(word, ahocs, make_singular=True, only_nouns=True)
            result1 = comp_split.merge_fractions(dissection1)
        except IndexError:
            # Если GCS падает, значит, он не нашел компонентов. Результат - пустой список.
            print("    -> !! Поймали баг в GCS (IndexError), результат пустой.")
            result1 = []
        print(f"  Режим 1 (По умолчанию, only_nouns=True, make_singular=True):")
        print(f"    -> Результат: {result1}")

        # --- Тест 2: Режим --gcs-only-nouns-false ---
        # В этом режиме бага нет, но для единообразия тоже добавим обработку
        result2 = []
        try:
            dissection2 = comp_split.dissect(word, ahocs, make_singular=True, only_nouns=False)
            result2 = comp_split.merge_fractions(dissection2)
        except IndexError:
            result2 = []
        print(f"  Режим 2 (--gcs-only-nouns-false):")
        print(f"    -> Результат: {result2}")

        # --- Тест 3: Режим --gcs-combine-noun-modes ---
        combined_result = sorted(list(set(result1) | set(result2)))
        print(f"  Режим 3 (Эмуляция --gcs-combine-noun-modes):")
        print(f"    -> Результат: {combined_result}")

        # --- Тест 4: Демонстрация make_singular=False ---
        result4 = []
        try:
            # ПАТЧ: И здесь тоже добавляем обработчик
            dissection4 = comp_split.dissect(word, ahocs, make_singular=False, only_nouns=True)
            result4 = comp_split.merge_fractions(dissection4)
        except IndexError:
            result4 = []
        print(f"  Режим 4 (only_nouns=True, make_singular=False):")
        print(f"    -> Результат: {result4}")
        
        print("-" * (len(word) + 20))

if __name__ == "__main__":
    run_test()

Загрузка словаря GCS...
Loading data file - U:/voothi/20241223170748-token-extraction/20250826000433-test/german.dic
Словарь загружен.

--- Анализ слова: 'Ausbildungserfahrung' ---
Dissect compound:  Ausbildungserfahrung
  Режим 1 (По умолчанию, only_nouns=True, make_singular=True):
    -> Результат: ['Ausbildung', 'Erfahrung']
Dissect compound:  Ausbildungserfahrung
  Режим 2 (--gcs-only-nouns-false):
    -> Результат: ['ausBildung', 's', 'Erfahrung']
  Режим 3 (Эмуляция --gcs-combine-noun-modes):
    -> Результат: ['Ausbildung', 'Erfahrung', 'ausBildung', 's']
Dissect compound:  Ausbildungserfahrung
  Режим 4 (only_nouns=True, make_singular=False):
    -> Результат: ['Ausbildungs', 'Erfahrung']
----------------------------------------
--- Анализ слова: 'Informationstechnik' ---
Dissect compound:  Informationstechnik
  Режим 1 (По умолчанию, only_nouns=True, make_singular=True):
    -> Результат: ['Info', 'rm', 'atIon', 'Technik']
Dissect compound:  Informationstechnik
  Режим 2 (--gc

In [6]:
# gcs_test.py (версия с детальным выводом параметров и результатов)
import german_compound_splitter.comp_split as comp_split
import os

# --- НАСТРОЙКИ ---
DICTIONARY_PATH = "U:/voothi/20241223170748-token-extraction/20250826000433-test/german.dic"

WORDS_TO_TEST = [
    "Ausbildungserfahrung",
    "DSL-Dienste",
    "Informationstechnik",
    "Arbeitspapiere",
    "Unternehmenssoftware"
]

# --- СКРИПТ ---

def run_single_test(word, ahocs, only_nouns_val, make_singular_val):
    """Выполняет один тест с заданными параметрами и печатает результат."""
    try:
        # 1. Получаем сырой результат разделения
        dissection = comp_split.dissect(word, ahocs, make_singular=make_singular_val, only_nouns=only_nouns_val)
        print(f"    -> Сырой результат (dissect): {dissection}")

        # 2. Получаем обработанный результат после слияния
        merged_result = comp_split.merge_fractions(dissection)
        print(f"    -> Результат (merge_fractions): {merged_result}")
        return merged_result

    except IndexError:
        print("    -> !! Поймали баг в GCS (IndexError), результат пустой.")
        return []

def run_all_tests():
    """Запускает изолированный тест GCS для всех слов и режимов."""
    if not os.path.exists(DICTIONARY_PATH):
        print(f"ОШИБКА: Файл словаря не найден по пути: {DICTIONARY_PATH}")
        return

    print("Загрузка словаря GCS...")
    ahocs = comp_split.read_dictionary_from_file(DICTIONARY_PATH)
    print("Словарь загружен.\n")

    for word in WORDS_TO_TEST:
        print(f"--- Анализ слова: '{word}' ---")
        
        # --- Тест 1 ---
        print("\n  Режим 1 (По умолчанию): only_nouns=True, make_singular=True")
        result1 = run_single_test(word, ahocs, only_nouns_val=True, make_singular_val=True)

        # --- Тест 2 ---
        print("\n  Режим 2 (--gcs-only-nouns-false): only_nouns=False, make_singular=True")
        result2 = run_single_test(word, ahocs, only_nouns_val=False, make_singular_val=True)

        # --- Тест 3 ---
        print("\n  Режим 3 (Эмуляция --gcs-combine-noun-modes): Объединение результатов Режима 1 и 2")
        combined_result = sorted(list(set(result1) | set(result2)))
        print(f"    -> Финальный результат: {combined_result}")

        # --- Тест 4 ---
        print("\n  Режим 4 (Отключен make_singular): only_nouns=True, make_singular=False")
        result4 = run_single_test(word, ahocs, only_nouns_val=True, make_singular_val=False)
        
        print("\n" + "=" * (len(word) + 20))


if __name__ == "__main__":
    run_all_tests()

Загрузка словаря GCS...
Loading data file - U:/voothi/20241223170748-token-extraction/20250826000433-test/german.dic
Словарь загружен.

--- Анализ слова: 'Ausbildungserfahrung' ---

  Режим 1 (По умолчанию): only_nouns=True, make_singular=True
Dissect compound:  Ausbildungserfahrung
    -> Сырой результат (dissect): ['Ausbildung', 'Erfahrung']
    -> Результат (merge_fractions): ['Ausbildung', 'Erfahrung']

  Режим 2 (--gcs-only-nouns-false): only_nouns=False, make_singular=True
Dissect compound:  Ausbildungserfahrung
    -> Сырой результат (dissect): ['aus', 'Bildung', 's', 'Erfahrung']
    -> Результат (merge_fractions): ['ausBildung', 's', 'Erfahrung']

  Режим 3 (Эмуляция --gcs-combine-noun-modes): Объединение результатов Режима 1 и 2
    -> Финальный результат: ['Ausbildung', 'Erfahrung', 'ausBildung', 's']

  Режим 4 (Отключен make_singular): only_nouns=True, make_singular=False
Dissect compound:  Ausbildungserfahrung
    -> Сырой результат (dissect): ['Ausbildungs', 'Erfahrung']

In [7]:
# gcs_test.py (финальная версия, все 4 режима)
import german_compound_splitter.comp_split as comp_split
import os

# --- НАСТРОЙКИ ---
DICTIONARY_PATH = "U:/voothi/20241223170748-token-extraction/20250826000433-test/german.dic"

WORDS_TO_TEST = [
    "Ausbildungserfahrung",
    "DSL-Dienste",
    "Informationstechnik",
    "Arbeitspapiere",
    "Unternehmenssoftware"
]

# --- СКРИПТ ---

def run_single_test(word, ahocs, only_nouns_val, make_singular_val):
    """Выполняет один тест с заданными параметрами и печатает результат."""
    try:
        # 1. Получаем сырой результат разделения
        dissection = comp_split.dissect(word, ahocs, make_singular=make_singular_val, only_nouns=only_nouns_val)
        print(f"    -> Сырой результат (dissect): {dissection}")

        # 2. Получаем обработанный результат после слияния
        merged_result = comp_split.merge_fractions(dissection)
        print(f"    -> Результат (merge_fractions): {merged_result}")
        return merged_result

    except IndexError:
        print("    -> !! Поймали баг в GCS (IndexError), результат пустой.")
        return []

def run_all_tests():
    """Запускает изолированный тест GCS для всех слов и всех 4 режимов."""
    if not os.path.exists(DICTIONARY_PATH):
        print(f"ОШИБКА: Файл словаря не найден по пути: {DICTIONARY_PATH}")
        return

    print("Загрузка словаря GCS...")
    ahocs = comp_split.read_dictionary_from_file(DICTIONARY_PATH)
    print("Словарь загружен.\n")

    for word in WORDS_TO_TEST:
        print(f"--- Анализ слова: '{word}' ---")
        
        # --- Тест 1: Стандартный режим ---
        print("\n  РЕЖИМ 1: only_nouns=True, make_singular=True (Стандартный/Безопасный)")
        run_single_test(word, ahocs, only_nouns_val=True, make_singular_val=True)

        # --- Тест 2: Безопасный, но без приведения к ед. числу ---
        print("\n  РЕЖИМ 2: only_nouns=True, make_singular=False (Безопасный, без приведения к ед. числу)")
        run_single_test(word, ahocs, only_nouns_val=True, make_singular_val=False)

        # --- Тест 3: Агрессивный режим с приведением к ед. числу ---
        print("\n  РЕЖИМ 3: only_nouns=False, make_singular=True (Агрессивный, с приведением к ед. числу)")
        run_single_test(word, ahocs, only_nouns_val=False, make_singular_val=True)

        # --- Тест 4: Максимально агрессивный режим ---
        print("\n  РЕЖИМ 4: only_nouns=False, make_singular=False (Максимально агрессивный, без приведения к ед. числу)")
        run_single_test(word, ahocs, only_nouns_val=False, make_singular_val=False)
        
        print("\n" + "=" * (len(word) + 20))


if __name__ == "__main__":
    run_all_tests()

Загрузка словаря GCS...
Loading data file - U:/voothi/20241223170748-token-extraction/20250826000433-test/german.dic
Словарь загружен.

--- Анализ слова: 'Ausbildungserfahrung' ---

  РЕЖИМ 1: only_nouns=True, make_singular=True (Стандартный/Безопасный)
Dissect compound:  Ausbildungserfahrung
    -> Сырой результат (dissect): ['Ausbildung', 'Erfahrung']
    -> Результат (merge_fractions): ['Ausbildung', 'Erfahrung']

  РЕЖИМ 2: only_nouns=True, make_singular=False (Безопасный, без приведения к ед. числу)
Dissect compound:  Ausbildungserfahrung
    -> Сырой результат (dissect): ['Ausbildungs', 'Erfahrung']
    -> Результат (merge_fractions): ['Ausbildungs', 'Erfahrung']

  РЕЖИМ 3: only_nouns=False, make_singular=True (Агрессивный, с приведением к ед. числу)
Dissect compound:  Ausbildungserfahrung
    -> Сырой результат (dissect): ['aus', 'Bildung', 's', 'Erfahrung']
    -> Результат (merge_fractions): ['ausBildung', 's', 'Erfahrung']

  РЕЖИМ 4: only_nouns=False, make_singular=False (М